# Crawling dan Scraping Data

# Crawling Data

Crawling data dilakukan untuk mengumpulkan URL berita yang akan digunakan pada proses scraping. Pada tahap ini, data diperoleh langsung dari halaman indeks berita Detik.com untuk kategori Sport dan Finance.

Proses crawling menggunakan library Requests untuk mengakses halaman indeks dan BeautifulSoup untuk membaca serta mengambil URL artikel dari struktur HTML halaman tersebut. URL berita yang berhasil dikumpulkan kemudian digunakan sebagai sumber pada tahap scraping untuk mengambil isi berita.

Pada proses ini, masing-masing kategori dikumpulkan sebanyak 100 URL berita, sehingga diperoleh total 200 URL yang terdiri dari 100 berita kategori Sport dan 100 berita kategori Finance.

## Pengambilan URL Berita

Library Requests digunakan untuk mengakses halaman indeks berita Detik.com, sedangkan BeautifulSoup digunakan untuk mengambil URL artikel.

URL yang diperoleh disimpan dalam dua list, yaitu link_sport dan link_finance. Masing-masing kategori dikumpulkan sebanyak 100 URL, sehingga total terdapat 200 URL berita.

In [3]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import time

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/154.0.0.0 Safari/537.36"
}

def ambil_link_berita(base_url, target=100):
    links = []

    for halaman in range(1, 15):
        url = f"{base_url}/indeks?page={halaman}"
        response = requests.get(url, headers=HEADERS, timeout=15)
        soup = BeautifulSoup(response.text, "html.parser")

        for a in soup.find_all("a", href=True):
            href = urljoin(base_url + "/", a["href"])

            if base_url in href and "/d-" in href and href not in links:
                links.append(href)

                if len(links) >= target:
                    break

        if len(links) >= target:
            break

        time.sleep(1)

    return links[:target]


link_sport = ambil_link_berita("https://sport.detik.com", 100)
link_finance = ambil_link_berita("https://finance.detik.com", 100)

print(f"Sport berhasil di-crawling: {len(link_sport)} berita")
print(f"Finance berhasil di-crawling: {len(link_finance)} berita")
print(f"Total berita berhasil di-crawling: {len(link_sport) + len(link_finance)} berita")

Sport berhasil di-crawling: 100 berita
Finance berhasil di-crawling: 100 berita
Total berita berhasil di-crawling: 200 berita


## Hasil Crawling

Hasil crawling berhasil memperoleh 100 URL berita dari kategori Sport dan 100 URL dari kategori Finance, sehingga terdapat 200 URL berita yang dikumpulkan.

URL tersebut kemudian digunakan sebagai sumber pada tahap scraping untuk mengambil isi berita. Dari URL yang tersedia, nantinya dipilih 100 data berita yang berhasil diekstrak dari masing-masing kategori.

# Scraping Isi Berita

URL hasil crawling selanjutnya digunakan untuk mengambil isi berita menggunakan library Trafilatura. Fungsi fetch_url() digunakan untuk mengambil halaman berita, kemudian extract() digunakan untuk mengekstrak teks utama.

Data yang berhasil diekstrak kemudian disimpan beserta label kategorinya, yaitu sport dan finance.

In [4]:
import trafilatura
import time
import random

def ekstrak_dataset(links, label, target=100):
    data = []
    gagal = []

    for link in links:
        if len(data) >= target:
            break

        try:
            downloaded = trafilatura.fetch_url(link)
            isi = None

            if downloaded:
                isi = trafilatura.extract(downloaded, favor_recall=True)

            if isi:
                data.append({"isi_berita": isi, "label": label})
            else:
                gagal.append(link)

        except Exception:
            gagal.append(link)

        print(f"[{label}] {len(data)}/{target}", end="\r")
        time.sleep(random.uniform(2, 5))

    print()
    return data, gagal


data_sport, gagal_sport = ekstrak_dataset(link_sport, "sport", 100)
data_finance, gagal_finance = ekstrak_dataset(link_finance, "finance", 100)

print(f"Data Sport Berhasil di Ekstrak: {len(data_sport)}, Gagal: {len(gagal_sport)}")
print(f"Data Finance Berhasil di Ekstrak: {len(data_finance)}, Gagal: {len(gagal_finance)}")
print(f"Total Data Berhasil di Ekstrak: {len(data_sport) + len(data_finance)}")

[sport] 100/100
[finance] 100/100
Data Sport Berhasil di Ekstrak: 100, Gagal: 0
Data Finance Berhasil di Ekstrak: 100, Gagal: 0
Total Data Berhasil di Ekstrak: 200


Berdasarkan hasil scraping, seluruh 100 berita Sport dan 100 berita Finance berhasil diekstrak.

# Pembentukan Dataset

In [6]:
import pandas as pd

data_sport = data_sport[:100]
data_finance = data_finance[:100]
df = pd.DataFrame(data_sport + data_finance)
df.insert(0, "id", range(1, len(df) + 1))

print(df.shape)
df.head()

(200, 3)


,id,isi_berita,label
0,1,Jepang - Austin Krajicek-Nikola Mektic berhasi...,sport
1,2,Jorge Martin sedang memperjuangkan titel juara...,sport
2,3,Rider Indonesia Veda Ega Pratama balapan di Ma...,sport
3,4,"Pelatih speed tim panjat tebing, Fitriyani, me...",sport
4,5,Pertamina Grand Prix of Indonesia 2026 sudah m...,sport


In [7]:
df.to_csv("dataset_berita.csv", index=False, encoding="utf-8")
print("Selesai")

Selesai


Pada tahap ini, data berita dari kategori Sport dan Finance masing-masing diambil sebanyak 100 data. Kedua data tersebut kemudian digabungkan menjadi satu dataset menggunakan Pandas. Setiap data diberi nomor ID secara berurutan, sehingga diperoleh 100 data berita dalam format CSV yang siap digunakan.